# Module B03 — Making Decisions

## Exercise 4: Four levels deep, then flat

Conditions nest. One `if` inside another is reasonable, two is common, four is
where a reader starts scrolling back up to work out which branch they are in.

Nesting is not wrong. It is expensive, and the cost is paid by whoever reads the
code next, which is usually you in three months. This notebook shows the cost,
then three ways of paying less of it, none of which change what the program
does.

| | |
|---|---|
| Time | About 40 minutes |
| You need | This notebook |
| Comes after | Exercise 3, four rules, one verdict |

---

## 1. What you are carrying in

`if` runs its block when the condition is true. `elif` gives the next
alternative and is tested only when everything above it failed. `else` catches
the rest. `and` is true only when both sides are, `or` when either is, and `not`
flips one condition.

One run, so this notebook stands on its own.

In [ ]:
logged_in = True
subscribed = False

if logged_in and not subscribed:
    print("logged in, no subscription")
elif logged_in:
    print("logged in and subscribed")
else:
    print("not logged in")

---

## 2. An `if` inside an `if`

The block belonging to an `if` can contain anything, including another `if`.
Each level adds four more spaces.

```
if logged_in:                    level 1
    if subscribed:               level 2, only reached when logged_in
        print("playing")         level 3, only reached when both
```

Run it, then change either name to `False` and run it again.

In [ ]:
logged_in = True
subscribed = True

if logged_in:
    if subscribed:
        print("playing")

---

## 3. What nesting costs

Here is the same decision at four levels, with a message for every outcome. Read
it, and count what you have to hold in your head to understand the line that
says `playing`.

In [ ]:
logged_in = True
subscribed = True
region_blocked = False
age = 21

if logged_in:
    if subscribed:
        if not region_blocked:
            if age >= 18:
                message = "playing"
            else:
                message = "this title is rated 18"
        else:
            message = "not available in your region"
    else:
        message = "renew your subscription"
else:
    message = "log in first"

print(message)

To read the line that says `playing` you must hold four facts at once: logged
in, subscribed, not blocked, old enough. To read the line that says
`renew your subscription` you must scroll up past eleven lines to find which
`if` that `else` belongs to.

The shape is the tell.

```
if
    if
        if
            if
                the actual work
            else
        else
    else
else
```

The work sits at the far right, and the branches that handle problems are spread
down the left in reverse order. Everything the reader needs is far from
everything else. This shape has a name in code review: the arrow, or the
staircase.

Nothing here is a bug. All four techniques below leave the behaviour exactly as
it is.

---

## 4. The indentation error that deep nesting produces

Levels this deep are where you meet the third indentation error. The next cell
fails on purpose.

As in exercise 1, the broken code is held in text and handed to `exec`, because
a cell containing a syntax error cannot run at all.

In [ ]:
staircase = '''if True:
    if True:
        print("level three")
      print("this line is at no level anybody declared")
'''

exec(staircase)

```
IndentationError: unindent does not match any outer indentation level
```

**`IndentationError`** again, with a different cause from exercise 1. There, a
block was missing. Here, a line came back out to a depth that no enclosing block
uses. Six spaces is not level one, which is four, and not level two, which is
eight. Python has no block to attach it to.

The message is one of the least helpful in Python, because it names the symptom
and not the line you should compare against. When you meet it, look at the line
in the traceback and at the lines above it, and check that every one starts at a
multiple of four.

Deep nesting makes this more likely, which is one more reason to prefer flat
code.

---

## 5. Technique one: `and` collapses levels that all must be true

When every level must be true for the work to happen, and you do not need to
know which one failed, `and` says the same thing on one line.

In [ ]:
logged_in = True
subscribed = True
region_blocked = False
age = 21

if logged_in and subscribed and not region_blocked and age >= 18:
    print("playing")
else:
    print("cannot play")

Four levels became one. The condition now reads as the sentence you would say
out loud.

What it gave up is the reason. This version cannot tell the user which of the
four things was wrong, only that something was. When the reason matters, the
next technique keeps it.

---

## 6. Technique two: handle the problems first, on the way in

Invert each condition and deal with the failure immediately. What is left at the
bottom is the case you actually care about, at one level of indentation.

This is called a **guard**: a short check at the top that disposes of a case so
the rest of the code does not have to think about it.

In [ ]:
logged_in = True
subscribed = False
region_blocked = False
age = 21

if not logged_in:
    message = "log in first"
elif not subscribed:
    message = "renew your subscription"
elif region_blocked:
    message = "not available in your region"
elif age < 18:
    message = "this title is rated 18"
else:
    message = "playing"

print(message)

Same four conditions, same five outcomes, same behaviour. Compare it with
section 3.

- Every branch is at the same indentation, so the reader is never lost.
- Each condition sits next to the message it produces, rather than eleven lines
  away.
- The order is the order you would say it in: log in, subscribe, region, age.
- Adding a fifth rule means adding one `elif`, not re-indenting the file.

The exercise 1 rule still applies. This is an `elif` chain, so the first true
branch wins and the rest are skipped. Order it the way you want the problems
reported.

In module B06 you meet functions and `return`, which lets a guard exit
immediately rather than continuing down a chain. The shape you are learning here
is the same one, and it is the one that matters.

---

## 7. Technique three: name the combined condition

When the condition is long, give it a name and let the `if` read as English.

In [ ]:
logged_in = True
subscribed = True
region_blocked = False
age = 21

can_play = logged_in and subscribed and not region_blocked and age >= 18

if can_play:
    print("playing")
else:
    print("cannot play")

`if can_play:` is the line a reader wants to meet. The definition above it is
there when they need the detail, and ignorable when they do not.

This combines with the guard ladder rather than competing with it. Name the
parts, then guard on the names.

---

## 8. The mistake nesting hides: an `else` at the wrong level

In Python, which `if` an `else` belongs to is decided entirely by its
indentation. Move it four spaces and it belongs to a different `if`, with no
error and no warning.

Both cells below run. They disagree.

In [ ]:
logged_in = True
subscribed = False
message = "nothing was decided"

if logged_in:
    if subscribed:
        message = "playing"
else:
    message = "log in first"

print(message)

In [ ]:
logged_in = True
subscribed = False
message = "nothing was decided"

if logged_in:
    if subscribed:
        message = "playing"
    else:
        message = "renew your subscription"
else:
    message = "log in first"

print(message)

The first cell prints `nothing was decided`. Its `else` belongs to the outer
`if`, so a logged-in user without a subscription falls through every branch and
nothing is assigned. Had `message` not been given a value beforehand, the
`print` would have raised a `NameError` instead, which would at least have told
you something was wrong.

The second cell differs by four spaces and prints the intended message.

This is exercise 1's lesson in a new costume: the failure is silent. A flat
guard ladder cannot produce it, because there is nowhere for an `else` to
attach wrongly.

---

# Your turn

**Do not delete the `# ANSWER n` marker lines.**

### Task 1

Predict the message for each of the three settings **before running**. The code
is the nested version from section 3, unchanged.

In [ ]:
# ANSWER 1
# Setting A: logged_in True,  subscribed True,  region_blocked True,  age 21 -> ___
# Setting B: logged_in True,  subscribed True,  region_blocked False, age 12 -> ___
# Setting C: logged_in False, subscribed False, region_blocked True,  age 12 -> ___

logged_in = True
subscribed = True
region_blocked = True
age = 21

if logged_in:
    if subscribed:
        if not region_blocked:
            if age >= 18:
                message = "playing"
            else:
                message = "this title is rated 18"
        else:
            message = "not available in your region"
    else:
        message = "renew your subscription"
else:
    message = "log in first"

print(message)

### Task 2

A different four-level nest, for a loan application. Flatten it using technique
one, so that the four levels become a single condition.

This version only has to answer yes or no.

In [ ]:
# ANSWER 2
# The nested original, for reference:
#
# if age >= 18:
#     if income >= 20000:
#         if not has_default:
#             if years_employed >= 1:
#                 approved = True
#             else:
#                 approved = False
#         else:
#             approved = False
#     else:
#         approved = False
# else:
#     approved = False

age = 22
income = 24000
has_default = False
years_employed = 2

approved = ___

print("approved:", approved)

### Task 3

Now the same rules, but the applicant is owed a reason. Rewrite it as a guard
ladder using technique two, so that every branch sits at one level and each
reason sits next to the condition that produced it.

Run it four more times, changing one value each time, and check that the reason
matches.

In [ ]:
# ANSWER 3
age = 22
income = 24000
has_default = True
years_employed = 2

if ___:
    reason = "applicants must be 18 or over"
elif ___:
    reason = "income must be at least 20000"
elif ___:
    reason = "a default is on record"
elif ___:
    reason = "at least one year of employment is required"
else:
    reason = "approved"

print(reason)

### Task 4

You now have two flattenings of the same rules: one condition in task 2, and a
guard ladder in task 3.

Say which you would maintain, and why. Then name the change to the requirements
that would flip your answer to the other one. Both versions are correct, so the
reasoning is the whole of the answer.

In [ ]:
# ANSWER 4
which_i_would_maintain_and_why = "___"
what_would_flip_my_answer = "___"

# One more line of judgement: what does the nested version in section 3 have
# that neither flat version has, if anything? ___

---

## Self-check

You do not need to understand this cell. It is machinery, not material.

In [ ]:
def _answer(marker):
    """Find the most recent cell you ran that contains the given marker."""
    try:
        matches = [c for c in _ih if marker in c and "def _answer" not in c]
    except NameError:
        print("Run this in Jupyter or VS Code so the self-check can see your cells.")
        return ""
    return matches[-1] if matches else ""


def check(passed, message):
    print(("PASS  " if passed else "FAIL  ") + message)
    return bool(passed)

NEXT = "Move on to exercise 5, the last in this module, where every case has to be covered."

a1, a2, a3, a4 = (_answer("# ANSWER 1"), _answer("# ANSWER 2"),
                  _answer("# ANSWER 3"), _answer("# ANSWER 4"))

results = [
    check("age 21 -> ___" not in a1 and a1.count("___") == 0,
          "Task 1: all three messages predicted before running"),
    check("___" not in a2, "Task 2: the four levels collapsed into one condition"),
    check(a2.count("and") >= 3 and "if " not in a2.split("approved = ")[-1],
          "Task 2: one condition joined by and, with no nesting left"),
    check("___" not in a3, "Task 3: every guard filled in"),
    check(a3.count("elif") >= 3 and "else:" in a3,
          "Task 3: a flat ladder with a final else"),
    check(a3.count("age") >= 2 and a3.count("income") >= 2
          and a3.count("has_default") >= 2 and a3.count("years_employed") >= 2,
          "Task 3: all four rules are guarded on, one branch each"),
    check(a4.count("___") == 0,
          "Task 4: you chose a version, gave a reason, and named what flips it"),
]

print()
failed = results.count(False)
print("%d of %d checks failing. Keep going." % (failed, len(results)) if failed
      else "All %d checks passing. %s" % (len(results), NEXT))

---

## What you learned

- An `if` can contain another `if`, and each level adds four spaces.
- Nesting costs the reader: the work ends up far right, and each `else` ends up
  far from the `if` it belongs to.
- Deep nesting produces `IndentationError: unindent does not match any outer
  indentation level`, which names the symptom rather than the cause.
- `and` collapses levels that all have to be true, at the cost of knowing which
  one failed.
- A guard ladder handles the problems first and leaves the real case flat at the
  bottom, keeping the reason for each outcome next to its condition.
- Naming the combined condition lets the `if` read as a sentence.
- Which `if` an `else` belongs to is decided by indentation alone, so a
  misplaced `else` changes the meaning silently.

## Before you move on

- [ ] You can draw the arrow shape and say why it is expensive.
- [ ] You rewrote a four-level nest two different ways without changing its
      behaviour.
- [ ] You read an `IndentationError` about an unindent and can say what Python
      was comparing against.
- [ ] You have a reason for preferring one of your two flattenings, and know
      what would change your mind.

**Next:** exercise 5, where the decision has two dimensions, every possible
input has to land somewhere, and `match` makes its first appearance.